In [1]:
!pip install "cdsapi>=0.7.7" xarray netCDF4 pandas numpy pyarrow openpyxl tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.9 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import getpass

cds_key = getpass.getpass("Enter your CDS API key: ")

cdsapirc = Path.home() / ".cdsapirc"

cdsapirc.write_text(
    "url: https://cds.climate.copernicus.eu/api\n"
    f"key: {cds_key}\n"
)

print("CDS API key file created at:", cdsapirc)

Enter your CDS API key: ··········
CDS API key file created at: /root/.cdsapirc


In [3]:
import cdsapi
from pathlib import Path

DATASET = "reanalysis-era5-single-levels"

TEST_DIR = Path("era5_atmosphere_data/test")
TEST_DIR.mkdir(parents=True, exist_ok=True)

client = cdsapi.Client()

test_request = {
    "product_type": ["reanalysis"],
    "variable": [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "mean_sea_level_pressure",
        "surface_pressure"
    ],
    "year": ["2020"],
    "month": ["01"],
    "day": [f"{d:02d}" for d in range(1, 32)],
    "time": [
        "00:00", "01:00", "02:00", "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00", "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00", "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00", "21:00", "22:00", "23:00"
    ],
    # CDS area format: [North, West, South, East]
    "area": [22, 80, 5, 98],
    "data_format": "netcdf",
    "download_format": "unarchived"
}

test_file = TEST_DIR / "TEST_ERA5_BOB_2020_01.nc"

client.retrieve(DATASET, test_request, str(test_file))

print("Test file downloaded:", test_file)

2026-05-16 18:15:06,569 INFO [2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).
INFO:ecmwf.datastores.legacy_client:[2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).
2026-05-16 18:15:07,682 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for

3f1210c1f185ad5b7e4d215f461b203e.nc:   0%|          | 0.00/24.6M [00:00<?, ?B/s]

Test file downloaded: era5_atmosphere_data/test/TEST_ERA5_BOB_2020_01.nc


In [4]:
import cdsapi
from pathlib import Path
from calendar import monthrange
import time

DATASET = "reanalysis-era5-single-levels"

RAW_DIR = Path("era5_atmosphere_data/raw_hourly")
RAW_DIR.mkdir(parents=True, exist_ok=True)

BASINS = {
    "bob": {
        "name": "Bay_of_Bengal",
        "area": [22, 80, 5, 98]   # [North, West, South, East]
    },
    "as": {
        "name": "Arabian_Sea",
        "area": [22, 60, 5, 75]
    }
}

YEARS = [2020, 2021, 2022, 2023, 2024]

VARIABLES = [
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "mean_sea_level_pressure",
    "surface_pressure"
]

HOURS = [
    "00:00", "01:00", "02:00", "03:00", "04:00", "05:00",
    "06:00", "07:00", "08:00", "09:00", "10:00", "11:00",
    "12:00", "13:00", "14:00", "15:00", "16:00", "17:00",
    "18:00", "19:00", "20:00", "21:00", "22:00", "23:00"
]

client = cdsapi.Client()


def download_era5_month(basin_key, year, month, max_retries=3):
    basin = BASINS[basin_key]

    days_in_month = monthrange(year, month)[1]
    days = [f"{day:02d}" for day in range(1, days_in_month + 1)]

    target_file = RAW_DIR / f"ERA5_{basin_key}_atmosphere_{year}_{month:02d}.nc"

    if target_file.exists() and target_file.stat().st_size > 1000:
        print("Already exists, skipping:", target_file.name)
        return target_file

    request = {
        "product_type": ["reanalysis"],
        "variable": VARIABLES,
        "year": [str(year)],
        "month": [f"{month:02d}"],
        "day": days,
        "time": HOURS,
        "area": basin["area"],
        "data_format": "netcdf",
        "download_format": "unarchived"
    }

    for attempt in range(1, max_retries + 1):
        try:
            print("\nDownloading:")
            print("Basin:", basin["name"])
            print("Year-Month:", f"{year}-{month:02d}")
            print("Output:", target_file)
            print("Attempt:", attempt)

            client.retrieve(DATASET, request, str(target_file))

            if target_file.exists() and target_file.stat().st_size > 1000:
                print("Downloaded successfully:", target_file.name)
                return target_file
            else:
                raise RuntimeError("Downloaded file is missing or too small.")

        except Exception as e:
            print("Download failed:", basin_key, year, month)
            print("Reason:", e)

            if attempt < max_retries:
                print("Retrying after 20 seconds...")
                time.sleep(20)
            else:
                print("Final failure:", basin_key, year, month)
                return None


downloaded_files = []
failed_downloads = []

for basin_key in BASINS:
    for year in YEARS:
        for month in range(1, 13):
            result = download_era5_month(basin_key, year, month)

            if result is not None:
                downloaded_files.append(result)
            else:
                failed_downloads.append((basin_key, year, month))

print("\nERA5 hourly atmospheric download completed.")
print("Downloaded files:", len(downloaded_files))
print("Failed downloads:", len(failed_downloads))

if failed_downloads:
    print("Failed list:")
    for item in failed_downloads:
        print(item)

2026-05-16 18:15:35,104 INFO [2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).
INFO:ecmwf.datastores.legacy_client:[2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).



Downloading:
Basin: Bay_of_Bengal
Year-Month: 2020-01
Output: era5_atmosphere_data/raw_hourly/ERA5_bob_atmosphere_2020_01.nc
Attempt: 1


2026-05-16 18:15:35,670 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

3f1210c1f185ad5b7e4d215f461b203e.nc:   0%|          | 0.00/24.6M [00:00<?, ?B/s]

Downloaded successfully: ERA5_bob_atmosphere_2020_01.nc

Downloading:
Basin: Bay_of_Bengal
Year-Month: 2020-02
Output: era5_atmosphere_data/raw_hourly/ERA5_bob_atmosphere_2020_02.nc
Attempt: 1


...

a91629d615dc80670cd8f1c2e121ef98.nc:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

Downloaded successfully: ERA5_as_atmosphere_2024_11.nc

Downloading:
Basin: Arabian_Sea
Year-Month: 2024-12
Output: era5_atmosphere_data/raw_hourly/ERA5_as_atmosphere_2024_12.nc
Attempt: 1


2026-05-16 19:20:01,905 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

edffbac2e0af3540f2712a194bc25411.nc:   0%|          | 0.00/19.7M [00:00<?, ?B/s]

Downloaded successfully: ERA5_as_atmosphere_2024_12.nc

ERA5 hourly atmospheric download completed.
Downloaded files: 120
Failed downloads: 0


In [5]:
from pathlib import Path

RAW_DIR = Path("era5_atmosphere_data/raw_hourly")

files = sorted(RAW_DIR.glob("*.nc"))

print("Total ERA5 raw files:", len(files))

for f in files[:20]:
    print(f.name, round(f.stat().st_size / (1024 * 1024), 2), "MB")

Total ERA5 raw files: 120
ERA5_as_atmosphere_2020_01.nc 19.64 MB
ERA5_as_atmosphere_2020_02.nc 18.4 MB
ERA5_as_atmosphere_2020_03.nc 19.77 MB
ERA5_as_atmosphere_2020_04.nc 19.03 MB
ERA5_as_atmosphere_2020_05.nc 19.96 MB
ERA5_as_atmosphere_2020_06.nc 19.3 MB
ERA5_as_atmosphere_2020_07.nc 20.14 MB
ERA5_as_atmosphere_2020_08.nc 19.53 MB
ERA5_as_atmosphere_2020_09.nc 19.13 MB
ERA5_as_atmosphere_2020_10.nc 19.77 MB
ERA5_as_atmosphere_2020_11.nc 19.24 MB
ERA5_as_atmosphere_2020_12.nc 19.96 MB
ERA5_as_atmosphere_2021_01.nc 19.85 MB
ERA5_as_atmosphere_2021_02.nc 17.8 MB
ERA5_as_atmosphere_2021_03.nc 19.92 MB
ERA5_as_atmosphere_2021_04.nc 19.02 MB
ERA5_as_atmosphere_2021_05.nc 20.11 MB
ERA5_as_atmosphere_2021_06.nc 18.8 MB
ERA5_as_atmosphere_2021_07.nc 19.68 MB
ERA5_as_atmosphere_2021_08.nc 19.57 MB


In [6]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("era5_atmosphere_data/raw_hourly")
PROCESSED_DIR = Path("era5_atmosphere_data/processed_daily")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

all_files = sorted(RAW_DIR.glob("*.nc"))

print("Total ERA5 files found:", len(all_files))

if len(all_files) == 0:
    raise FileNotFoundError(
        "No ERA5 .nc files found in era5_atmosphere_data/raw_hourly. "
        "Please complete the full download first."
    )


def infer_basin_from_filename(filename):
    filename = filename.lower()

    if "_bob_" in filename:
        return "Bay_of_Bengal"
    elif "_as_" in filename:
        return "Arabian_Sea"
    else:
        return "Unknown"


def calculate_wind_direction(u, v):
    """
    Meteorological wind direction:
    Direction FROM which the wind is blowing, in degrees.
    """
    return (270 - np.degrees(np.arctan2(v, u))) % 360


def process_one_era5_file(file_path):
    file_path = Path(file_path)
    basin = infer_basin_from_filename(file_path.name)

    print("\nProcessing:", file_path.name)

    ds = xr.open_dataset(file_path)

    # Convert to DataFrame
    df = ds.to_dataframe().reset_index()

    # Standardize time column
    if "valid_time" in df.columns:
        df = df.rename(columns={"valid_time": "timestamp"})
    elif "time" in df.columns:
        df = df.rename(columns={"time": "timestamp"})
    else:
        ds.close()
        raise ValueError("No time column found. Available columns: " + str(df.columns.tolist()))

    # Standardize latitude / longitude
    if "latitude" in df.columns:
        df = df.rename(columns={"latitude": "lat"})

    if "longitude" in df.columns:
        df = df.rename(columns={"longitude": "lon"})

    # Some NetCDF files may include expver. It is not needed for our analysis.
    if "expver" in df.columns:
        df = df.drop(columns=["expver"])

    required_cols = ["timestamp", "lat", "lon", "u10", "v10", "msl", "sp"]
    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        print("Missing columns:", missing)
        print("Available columns:", df.columns.tolist())
        ds.close()
        raise ValueError("Required ERA5 columns missing.")

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["date"] = df["timestamp"].dt.date
    df["basin"] = basin

    # Clean numeric values
    for col in ["lat", "lon", "u10", "v10", "msl", "sp"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["timestamp", "lat", "lon", "u10", "v10", "msl", "sp"])

    # Derived features
    df["wind_speed"] = np.sqrt(df["u10"] ** 2 + df["v10"] ** 2)
    df["wind_direction"] = calculate_wind_direction(df["u10"], df["v10"])

    # Pressure conversion: Pa to hPa
    df["mean_sea_level_pressure_hpa"] = df["msl"] / 100.0
    df["surface_pressure_hpa"] = df["sp"] / 100.0

    daily = (
        df.groupby(["basin", "date", "lat", "lon"], as_index=False)
        .agg(
            u10_mean=("u10", "mean"),
            v10_mean=("v10", "mean"),
            wind_speed_mean=("wind_speed", "mean"),
            wind_speed_max=("wind_speed", "max"),
            wind_speed_std=("wind_speed", "std"),
            wind_direction_mean=("wind_direction", "mean"),
            mean_sea_level_pressure_mean=("mean_sea_level_pressure_hpa", "mean"),
            mean_sea_level_pressure_min=("mean_sea_level_pressure_hpa", "min"),
            surface_pressure_mean=("surface_pressure_hpa", "mean"),
            surface_pressure_min=("surface_pressure_hpa", "min"),
            valid_hourly_records=("wind_speed", "count")
        )
    )

    ds.close()

    print("Processed rows:", len(daily))

    return daily


daily_frames = []
failed_processing = []

for file_path in all_files:
    try:
        daily_df = process_one_era5_file(file_path)
        daily_frames.append(daily_df)
    except Exception as e:
        print("Skipped:", file_path.name)
        print("Reason:", e)
        failed_processing.append((file_path.name, str(e)))

if len(daily_frames) == 0:
    raise ValueError("No ERA5 files were processed successfully.")

era5_daily = pd.concat(daily_frames, ignore_index=True)

print("\nera5_daily created successfully.")
print("Shape:", era5_daily.shape)

era5_daily.head()

Total ERA5 files found: 120

Processing: ERA5_as_atmosphere_2020_01.nc
Processed rows: 130479

Processing: ERA5_as_atmosphere_2020_02.nc
Processed rows: 122061

Processing: ERA5_as_atmosphere_2020_03.nc
Processed rows: 130479

Processing: ERA5_as_atmosphere_2020_04.nc
Processed rows: 126270

Processing: ERA5_as_atmosphere_2020_05.nc
Processed rows: 130479

Processing: ERA5_as_atmosphere_2020_06.nc
Processed rows: 126270

Processing: ERA5_as_atmosphere_2020_07.nc
Processed rows: 130479

Processing: ERA5_as_atmosphere_2020_08.nc
Processed rows: 130479

Processing: ERA5_as_atmosphere_2020_09.nc
Processed rows: 126270

Processing: ERA5_as_atmosphere_2020_10.nc
Processed rows: 130479

Processing: ERA5_as_atmosphere_2020_11.nc
Processed rows: 126270

Processing: ERA5_as_atmosphere_2020_12.nc
Processed rows: 130479

Processing: ERA5_as_atmosphere_2021_01.nc
Processed rows: 130479

Processing: ERA5_as_atmosphere_2021_02.nc
Processed rows: 117852

Processing: ERA5_as_atmosphere_2021_03.nc
Proce

,basin,date,lat,lon,u10_mean,v10_mean,wind_speed_mean,wind_speed_max,wind_speed_std,wind_direction_mean,mean_sea_level_pressure_mean,mean_sea_level_pressure_min,surface_pressure_mean,surface_pressure_min,valid_hourly_records
0,Arabian_Sea,2020-01-01,5.0,60.00,-4.675491,-2.666956,5.427516,6.099535,0.408919,60.335678,1011.757630,1009.888125,1011.764557,1009.896875,24
1,Arabian_Sea,2020-01-01,5.0,60.25,-4.491449,-2.624231,5.243874,5.941792,0.434406,59.628876,1011.772943,1009.878125,1011.779141,1009.883750,24
2,Arabian_Sea,2020-01-01,5.0,60.50,-4.314773,-2.599532,5.081857,5.851656,0.472084,58.692509,1011.777526,1009.833125,1011.762474,1009.813750,24
3,Arabian_Sea,2020-01-01,5.0,60.75,-4.224115,-2.569585,4.991982,5.718002,0.514333,58.226780,1011.779193,1009.773125,1011.836641,1009.833750,24
4,Arabian_Sea,2020-01-01,5.0,61.00,-4.219924,-2.516850,4.967691,5.814530,0.527341,58.611431,1011.781693,1009.743125,1011.854141,1009.813750,24


In [ ]:
from pathlib import Path

PROCESSED_DIR = Path("era5_atmosphere_data/processed_daily")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

final_csv = PROCESSED_DIR / "era5_daily_grid_bob_as_2020_2024.csv"
final_parquet = PROCESSED_DIR / "era5_daily_grid_bob_as_2020_2024.parquet"

era5_daily.to_csv(final_csv, index=False)
era5_daily.to_parquet(final_parquet, index=False)

print("Saved full CSV:", final_csv)
print("Saved full Parquet:", final_parquet)
print("Rows:", len(era5_daily))

In [ ]:
summary = (
    era5_daily.groupby(["basin", "date"], as_index=False)
    .agg(
        u10_mean=("u10_mean", "mean"),
        v10_mean=("v10_mean", "mean"),
        wind_speed_mean=("wind_speed_mean", "mean"),
        wind_speed_max=("wind_speed_max", "max"),
        wind_speed_std=("wind_speed_mean", "std"),
        wind_direction_mean=("wind_direction_mean", "mean"),
        mean_sea_level_pressure_mean=("mean_sea_level_pressure_mean", "mean"),
        mean_sea_level_pressure_min=("mean_sea_level_pressure_min", "min"),
        surface_pressure_mean=("surface_pressure_mean", "mean"),
        surface_pressure_min=("surface_pressure_min", "min"),
        valid_grid_cells=("valid_hourly_records", "count"),
        total_hourly_records=("valid_hourly_records", "sum")
    )
)

summary_csv = PROCESSED_DIR / "era5_daily_basin_summary_2020_2024.csv"
summary_excel = PROCESSED_DIR / "era5_daily_basin_summary_2020_2024.xlsx"
summary_parquet = PROCESSED_DIR / "era5_daily_basin_summary_2020_2024.parquet"

summary.to_csv(summary_csv, index=False)
summary.to_excel(summary_excel, index=False)
summary.to_parquet(summary_parquet, index=False)

print("Saved summary CSV:", summary_csv)
print("Saved summary Excel:", summary_excel)
print("Saved summary Parquet:", summary_parquet)

summary.head()

In [12]:
import shutil
from google.colab import files

zip_name = "era5_atmosphere_bob_as_2020_2024_processed"

shutil.make_archive(zip_name, "zip", "era5_atmosphere_data/processed_daily")

files.download(zip_name + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import shutil
from google.colab import files

zip_name = "era5_atmosphere_bob_as_2020_2024_raw_hourly"

shutil.make_archive(zip_name, "zip", "era5_atmosphere_data/raw_hourly")

files.download(zip_name + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>